# AutoGen: um residente e um infectologista revisam pareceres da CCIH

O **AutoGen**, da Microsoft, é um framework para montar sistemas de **agentes que conversam entre si**. Cada agente é um modelo de linguagem com um papel, definido pela sua instrução de sistema; o AutoGen organiza a conversa: quem fala, em que ordem e quando parar.

Neste notebook, dois agentes trabalham como na **Comissão de Controle de Infecção Hospitalar (CCIH)**:

- o **residente** recebe uma situação do dia a dia (uma hemocultura, um antibiograma, um indicador do mês) e dá um **parecer rápido**, pela primeira impressão;
- o **infectologista** confere o parecer contra o **protocolo da CCIH**. Se encontrar um erro, aponta a regra que não foi seguida e o residente refaz; se estiver certo, escreve **APROVADO** e a conversa termina.

É o padrão **executor e revisor**: um agente produz, o outro confere com as regras, até chegarem a um parecer correto.

## 0. Como rodar este notebook

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pasteurjr/notebookvideoaulas/blob/main/06_agentes_autogen/notebook/agentes_autogen_v2_parecer_ccih.ipynb)

O curso tem duas partes, usadas por **todos** os notebooks:

| O quê | Onde |
|---|---|
| **Código** (notebooks, arquivos de configuração) | GitHub: **https://github.com/pasteurjr/notebookvideoaulas** |
| **Dados** (imagens e arquivos grandes) | Google Drive: **[pasta notebooksvideos](https://drive.google.com/drive/folders/1mMRjinjohj_jyMsJzO7rxgYKm6unVKze?usp=sharing)**, com a mesma estrutura de pastas do GitHub |

**Esta aula usa:** nenhum dado externo (só o código do GitHub); não precisa de GPU; chaves de API: `DEEPSEEK_API_KEY`.

### Opção A: Google Colab (recomendado, nada para instalar no computador)

1. **Adicione a pasta de dados do curso ao seu Google Drive** (uma vez só, vale para todas as aulas): abra o link **[notebooksvideos](https://drive.google.com/drive/folders/1mMRjinjohj_jyMsJzO7rxgYKm6unVKze?usp=sharing)**, clique com o botão direito no nome da pasta e escolha **Organizar → Adicionar atalho** (ou **Adicionar atalho ao Drive**) → **Meu Drive**. A pasta passa a aparecer como `MyDrive/notebooksvideos`, sem ocupar o seu espaço.
2. **Abra este notebook no Colab** pelo botão **Abrir no Colab** acima (ele abre direto do GitHub).
3. **Guarde as chaves de API nos *Secrets* do Colab:** clique no ícone de **chave (🔑)** na barra lateral esquerda, crie `DEEPSEEK_API_KEY` com as **suas** chaves (com crédito) e ative **Acesso ao notebook** em cada uma. As chaves ficam na sua conta Google, nunca no notebook.
4. **Rode a célula de preparação logo abaixo.** Ela clona o repositório do GitHub para o Colab, verifica quais bibliotecas a sessão do Colab já tem e **instala só as que faltarem**, e carrega as chaves dos *Secrets*.
5. Depois: **Ambiente de execução → Executar tudo**. Rodar a célula de preparação de novo não causa problema.

**Se aparecer erro de biblioteca** (`ModuleNotFoundError`, `ImportError` ou erro de versão): crie uma célula nova, rode o comando abaixo, depois vá em **Ambiente de execução → Reiniciar sessão** e rode a célula de preparação de novo.

```
!pip install autogen-agentchat autogen-ext[openai] python-dotenv
```

### Opção B: no seu computador

Pré-requisito: [Miniconda](https://docs.conda.io/en/latest/miniconda.html) (ou Anaconda) e Git instalados. No terminal:

```bash
git clone https://github.com/pasteurjr/notebookvideoaulas.git
cd notebookvideoaulas
conda create -n noteautogen python=3.12 -y
conda activate noteautogen
pip install autogen-agentchat "autogen-ext[openai]" python-dotenv jupyter
cd 06_agentes_autogen/notebook
jupyter notebook agentes_autogen_v2_parecer_ccih.ipynb
```

**Chaves:** copie `.env.exemplo` para `.env` e coloque nele as **suas** chaves (`DEEPSEEK_API_KEY`). O `.env` nunca vai para o GitHub.

No computador local, a célula de preparação detecta que não está no Colab e não faz nada.

In [1]:
# PREPARAÇÃO DO AMBIENTE NO GOOGLE COLAB (igual em todos os notebooks do curso; no computador local não faz nada)
AULA = "06_agentes_autogen/notebook"                           # pasta desta aula no repositório
USA_DADOS_DO_DRIVE = False                                     # copia os dados da pasta "notebooksvideos" do Drive
PACOTES = {"autogen_agentchat": "autogen-agentchat", "autogen_ext": "autogen-ext[openai]", "dotenv": "python-dotenv"}   # módulo: pacote pip
SECRETS = ["DEEPSEEK_API_KEY"]                                   # chaves lidas dos Secrets do Colab (ícone 🔑)

import os, sys, shutil, subprocess, importlib.util

if "google.colab" in sys.modules:
    # 1) Código: clona o repositório do GitHub para o Colab
    if not os.path.exists("/content/notebookvideoaulas"):
        subprocess.run(["git", "clone", "-q", "https://github.com/pasteurjr/notebookvideoaulas.git",
                        "/content/notebookvideoaulas"], check=True)
    os.chdir(f"/content/notebookvideoaulas/{AULA}")

    # 2) Dados: copia da pasta "notebooksvideos" (atalho no Meu Drive) para a pasta da aula
    if USA_DADOS_DO_DRIVE and not os.path.exists("dados"):
        from google.colab import drive
        drive.mount("/content/drive")
        shutil.copytree(f"/content/drive/MyDrive/notebooksvideos/{AULA}/dados", "dados")

    # 3) Bibliotecas: instala só as que a sessão do Colab ainda não tem
    faltando = [pip for modulo, pip in PACOTES.items() if importlib.util.find_spec(modulo) is None]
    if faltando:
        print("Instalando:", ", ".join(faltando))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltando], check=True)
        print("Pronto. Se o Colab pedir 'Reiniciar sessão', reinicie e rode esta célula de novo.")
    else:
        print("Todas as bibliotecas já estão instaladas no Colab.")

    # 4) Chaves de API: vêm dos Secrets do Colab (nunca escreva a chave no notebook)
    if SECRETS:
        from google.colab import userdata
        for chave in SECRETS:
            os.environ[chave] = userdata.get(chave)

    print("Pasta da aula:", os.getcwd())
    print("Conteúdo:", sorted(os.listdir(".")))
else:
    print("Ambiente local detectado: usando a pasta atual.")

Ambiente local detectado: usando a pasta atual.


## 1. O modelo de linguagem

Os dois agentes usam o **DeepSeek** (`deepseek-flash`), que segue a mesma interface da OpenAI: basta trocar o endereço (`base_url`). Desligamos o raciocínio interno do modelo (*thinking*): as respostas ficam diretas e a conversa, mais rápida. A chave fica no arquivo **`.env`** (no computador) ou nos **Secrets** do Colab, **nunca** no notebook. Crie uma chave **com crédito** em https://platform.deepseek.com/api_keys e coloque em `DEEPSEEK_API_KEY`.

In [2]:
import os, textwrap, warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient

load_dotenv()
modelo = OpenAIChatCompletionClient(
    model="deepseek-flash",                        # modelo atual do DeepSeek
    base_url="https://api.deepseek.com",
    api_key=os.environ["DEEPSEEK_API_KEY"],
    model_info={"vision": False, "function_calling": True, "json_output": True,
                "structured_output": True, "family": "unknown"},
    extra_body={"thinking": {"type": "disabled"}},   # sem raciocínio interno: respostas diretas
)
print("Modelo: deepseek-flash | chave carregada:", bool(os.environ.get("DEEPSEEK_API_KEY")))

Modelo: deepseek-flash | chave carregada: True


## 2. O protocolo da CCIH

O infectologista confere cada parecer contra estas regras, as mesmas da base de conhecimento da aula de RAG. O residente **não** recebe o protocolo: ele responde com o que sabe, como numa primeira avaliação.

In [3]:
PROTOCOLO = """
R1. IPCS associada a cateter central: infecção de corrente sanguínea PRIMÁRIA, isto é, que NÃO é secundária
    a outro foco identificado (como pneumonia ou infecção urinária com o mesmo microrganismo), em paciente
    com cateter venoso central há mais de 2 dias.
R2. Multirresistência (vigilância): contar as CLASSES de antimicrobianos com resultado R. O resultado I NÃO
    conta como resistência. Resistência em 3 ou mais classes caracteriza microrganismo multirresistente.
R3. Densidade de incidência = número de infecções / cateter-dia x 1000. Para comparar meses, compare as
    densidades, não o número bruto de casos.
R4. Precauções: multirresistentes e Clostridioides difficile, contato; tuberculose pulmonar, sarampo e
    varicela, aerossóis (respirador N95/PFF2); influenza e doença meningocócica, gotículas.
"""
print(PROTOCOLO)


R1. IPCS associada a cateter central: infecção de corrente sanguínea PRIMÁRIA, isto é, que NÃO é secundária
    a outro foco identificado (como pneumonia ou infecção urinária com o mesmo microrganismo), em paciente
    com cateter venoso central há mais de 2 dias.
R2. Multirresistência (vigilância): contar as CLASSES de antimicrobianos com resultado R. O resultado I NÃO
    conta como resistência. Resistência em 3 ou mais classes caracteriza microrganismo multirresistente.
R3. Densidade de incidência = número de infecções / cateter-dia x 1000. Para comparar meses, compare as
    densidades, não o número bruto de casos.
R4. Precauções: multirresistentes e Clostridioides difficile, contato; tuberculose pulmonar, sarampo e
    varicela, aerossóis (respirador N95/PFF2); influenza e doença meningocócica, gotículas.



## 3. Os dois agentes e uma ferramenta

Cada agente é um `AssistantAgent`: um nome, o modelo e uma **instrução de sistema**, que define o papel dele. O residente responde num formato fixo (**PARECER** e **JUSTIFICATIVA**); o infectologista recebe o protocolo e decide se aprova.

Modelos de linguagem erram contas. Por isso o infectologista tem uma **ferramenta**: uma função Python comum, que o AutoGen oferece ao modelo. Quando precisa conferir uma densidade de incidência, o agente **chama a função** em vez de fazer a conta de cabeça.

In [4]:
def densidade_incidencia(infeccoes: int, cateter_dia: int) -> str:
    """Calcula a densidade de incidência de IPCS por 1000 cateter-dia (regra R3)."""
    return f"{infeccoes} / {cateter_dia} x 1000 = {infeccoes / cateter_dia * 1000:.1f} por 1000 cateter-dia"

residente =AssistantAgent("residente", model_client=modelo, system_message="""
Você é residente de infectologia e ajuda a CCIH. Dê um parecer rápido, pela sua primeira impressão.
Responda com EXATAMENTE duas linhas, sem nada antes nem depois:
PARECER: a conclusão, em uma frase
JUSTIFICATIVA: o motivo, em uma ou duas frases
Se o infectologista apontar um erro, refaça o parecer com as mesmas duas linhas.""")

infectologista = AssistantAgent("infectologista", model_client=modelo, tools=[densidade_incidencia],
                                reflect_on_tool_use=True, system_message=f"""
Você é o infectologista da CCIH e revisa os pareceres do residente usando SOMENTE este protocolo:
{PROTOCOLO}
Leia com atenção o que o residente escreveu. Para conferir densidades, use SEMPRE a ferramenta densidade_incidencia. Se o parecer contrariar o protocolo, diga em até 3 frases qual regra (R1 a R4) não foi seguida e por quê,
sem dar a resposta pronta e sem escrever a palavra de aprovação.
Se o parecer estiver correto, confirme em uma frase, citando a regra, e termine com a palavra APROVADO.""")

print("Agentes:", residente.name, "e", infectologista.name)

Agentes: residente e infectologista


## 4. A equipe e a regra de parada

O `RoundRobinGroupChat` faz os agentes falarem **em turnos**: residente, infectologista, residente... A conversa termina quando o **infectologista** escreve **APROVADO** ou, por segurança, depois de **6 mensagens**.

In [5]:
parada = TextMentionTermination("APROVADO", sources=["infectologista"]) | MaxMessageTermination(6)
equipe = RoundRobinGroupChat([residente, infectologista], termination_condition=parada)

async def revisar(caso):
    await equipe.reset()                      # cada caso começa uma conversa nova
    resultado = await equipe.run(task=caso)
    print("CASO:", textwrap.fill(caso, 110), "\n")
    falas = [m for m in resultado.messages if m.type == "TextMessage" and m.source != "user"]   # só as falas dos agentes
    for msg in resultado.messages:
        if msg.type == "ToolCallExecutionEvent":            # o agente chamou a ferramenta
            for r in msg.content:
                print(f"   [ferramenta chamada pelo {msg.source}] {r.content}\n")
        elif msg in falas:
            print(f"[{msg.source}]")
            print("\n".join(textwrap.fill(linha, 110) for linha in msg.content.splitlines()), "\n")
    print("Falas:", len(falas), "| motivo da parada:", resultado.stop_reason)

## 5. Caso 1: é IPCS associada a cateter?

Hemocultura positiva em paciente com cateter central. A primeira impressão é classificar como infecção do cateter; mas há outro foco com o mesmo germe.

In [6]:
await revisar("Paciente na UTI com cateter venoso central há 5 dias tem hemocultura positiva para Klebsiella "
             "pneumoniae. Ele também tem pneumonia associada à ventilação mecânica, com a mesma Klebsiella no "
             "aspirado traqueal. Devemos notificar como IPCS associada a cateter central?")

CASO: Paciente na UTI com cateter venoso central há 5 dias tem hemocultura positiva para Klebsiella pneumoniae. Ele
também tem pneumonia associada à ventilação mecânica, com a mesma Klebsiella no aspirado traqueal. Devemos
notificar como IPCS associada a cateter central? 

[residente]
PARECER: Não notificar como IPCS associada a cateter central; trata-se de bacteremia secundária à pneumonia
associada à ventilação mecânica.
JUSTIFICATIVA: A mesma Klebsiella pneumoniae isolada no aspirado traqueal indica foco pulmonar como origem da
corrente sanguínea, e a presença de CVC por 5 dias não define infecção primária de corrente sanguínea quando
há foco secundário documentado. 

[infectologista]
O parecer está correto: pela R1, a IPCS associada a cateter central exige infecção de corrente sanguínea
primária, e aqui há foco secundário pulmonar documentado com o mesmo microrganismo, o que exclui a
notificação. APROVADO. 

Falas: 2 | motivo da parada: Text 'APROVADO' mentioned


## 6. Caso 2: é multirresistente?

Um antibiograma com dois resultados R e um resultado I. Quantas classes contam?

In [7]:
await revisar("Antibiograma de Klebsiella pneumoniae: ceftriaxona R, ciprofloxacino R, meropenem I e "
             "amicacina S. Pela vigilância, ela é multirresistente?")

CASO: Antibiograma de Klebsiella pneumoniae: ceftriaxona R, ciprofloxacino R, meropenem I e amicacina S. Pela
vigilância, ela é multirresistente? 

[residente]
PARECER: Sim, é multirresistente (MDR), pois é resistente a pelo menos um agente em três ou mais categorias de
antimicrobianos.
JUSTIFICATIVA: Há resistência a cefalosporinas (ceftriaxona), fluoroquinolonas (ciprofloxacino) e
carbapenêmicos (meropenem intermediário), além de sensibilidade apenas à amicacina. 

[infectologista]
A regra R2 foi contrariada: o resultado intermediário (meropenem I) não conta como resistência, então só há 2
classes com R (ceftriaxona e ciprofloxacino), não 3. 

[residente]
PARECER: Não, não é multirresistente (MDR), pois há resistência apenas a duas classes de antimicrobianos.
JUSTIFICATIVA: O meropenem intermediário não conta como resistência pela regra R2, restando apenas ceftriaxona
(cefalosporina) e ciprofloxacino (fluoroquinolona) como classes resistentes. 

[infectologista]
Correto: conforme R2,

## 7. Caso 3: a infecção aumentou?

Mais casos no mês, mas também muito mais dias de uso de cateter. O número bruto engana; aqui o infectologista usa a ferramenta para conferir as densidades.

In [8]:
await revisar("Na UTI, em agosto, houve 4 IPCS associadas a cateter central em 800 cateter-dia. Em setembro, "
             "houve 6 IPCS em 1.500 cateter-dia. A infecção aumentou de agosto para setembro?")

CASO: Na UTI, em agosto, houve 4 IPCS associadas a cateter central em 800 cateter-dia. Em setembro, houve 6 IPCS em
1.500 cateter-dia. A infecção aumentou de agosto para setembro? 

[residente]
PARECER: Não, a taxa de IPCS diminuiu de agosto para setembro.
JUSTIFICATIVA: Em agosto a densidade de incidência foi 5,0 por 1.000 cateter-dia, enquanto em setembro foi 4,0
por 1.000 cateter-dia; o número absoluto maior não indica aumento quando ajustado pelo denominador. 

   [ferramenta chamada pelo infectologista] 4 / 800 x 1000 = 5.0 por 1000 cateter-dia

   [ferramenta chamada pelo infectologista] 6 / 1500 x 1000 = 4.0 por 1000 cateter-dia

[infectologista]
As densidades conferem: 5,0 e 4,0 por 1.000 cateter-dia, de modo que o parecer está de acordo com a R3
(comparar densidades, não o número bruto). APROVADO 

Falas: 2 | motivo da parada: Text 'APROVADO' mentioned


## 8. Conclusão

- No **AutoGen**, a unidade de trabalho é a **conversa**: agentes com papéis trocam mensagens até uma **condição de parada**.
- O padrão **executor e revisor** funciona bem quando há **regras explícitas**: aqui, o protocolo da CCIH dado ao revisor.
- **Ferramentas** dão ao agente o que o modelo faz mal: contas exatas, consultas a bancos de dados, chamadas a sistemas.
- As situações escolhidas são armadilhas comuns na vigilância: infecção secundária, resultado I no antibiograma e comparação por número bruto em vez de densidade.
- **CrewAI ou AutoGen?** No CrewAI, que vimos em outra aula, definimos **tarefas** em sequência; no AutoGen, definimos **quem conversa com quem** e **quando a conversa termina**.
- Os agentes ajudam, mas o parecer final da CCIH continua sendo responsabilidade de um profissional.

**Experimente:** crie casos novos (uma indicação de precaução, uma investigação de surto), acrescente uma regra ao protocolo ou um terceiro agente que redige a notificação final.